# accel-sim silicon anchor — chained training-step calibration

Every earlier GEMM-level anchor here (`profile_step.py`) timed each GEMM
**in isolation** — its own tensor, its own optimizer, its own timing loop —
and summed the results to approximate a "step." A real model never trains
that way: it chains layers through one autograd graph. Comparing that
isolated-sum ground truth against a real chained measurement of the same
shapes showed real chained backward runs roughly **2x faster** than the
isolated sum on a T4 — which means `compute_efficiency`/`util_tiles` (fit
against the isolated sum) have been calibrated against the wrong target.

This notebook measures **real chained** forward+backward+Adam across 4
configs (the full 6-GEMM GPT-2 block, a wide-shallow chain, a narrow-deep
chain, and a small-batch run) so `compare_chain_grid.py` can refit those
two constants against a realistic training step instead.

**Before running:** `Runtime > Change runtime type > T4 GPU`, then
`Runtime > Run all`.

Writes `chain_grid_profile.json`, prints it, and auto-downloads it. Bring
that file back and run:

```bash
python validate/silicon/compare_chain_grid.py chain_grid_profile.json
```


In [ ]:
# ---- config (edit if you want) ---------------------------------------------
ITERS  = 50
WARMUP = 15
OUT    = "chain_grid_profile.json"

# Each config is a real chained forward+backward+Adam step, never timed in
# isolation. "gpt2_block_full" matches simulator/workloads.py's GPT2_BLOCK
# shapes exactly -- the original isolated-GEMM anchor's own shapes -- so
# it's a direct apples-to-apples replacement for that anchor's ground truth.
CONFIGS = {
    "gpt2_block_full": {
        "dims": [(768, 768), (768, 768), (768, 768), (768, 768),
                 (768, 3072), (3072, 768)],
        "tokens": 8 * 1024,
    },
    "wide_shallow": {
        "dims": [(768, 4096), (4096, 768)],
        "tokens": 8 * 1024,
    },
    "narrow_deep": {
        "dims": [(512, 512)] * 8,
        "tokens": 8 * 1024,
    },
    "small_batch": {
        "dims": [(768, 768), (768, 768), (768, 3072), (3072, 768)],
        "tokens": 1024,
    },
}


In [ ]:
import torch
assert torch.cuda.is_available(), "no CUDA device -- Runtime > Change runtime type > T4 GPU"

device = torch.device("cuda")
dtype = torch.float16
gpu = torch.cuda.get_device_name(0)
print(f"GPU: {gpu}   dtype=fp16   iters={ITERS} (+{WARMUP} warmup)")


In [ ]:
import statistics
import torch.nn as nn

def bench(dims, M, iters, warmup):
    layers = [nn.Linear(k, n, bias=False).to(device=device, dtype=dtype)
              for k, n in dims]
    params = [p for l in layers for p in l.parameters()]
    opt = torch.optim.Adam(params, lr=1e-4)
    x = torch.randn(M, dims[0][0], device=device, dtype=dtype, requires_grad=True)

    fwd, bwd, optt = [], [], []
    for i in range(warmup + iters):
        ev = [torch.cuda.Event(enable_timing=True) for _ in range(5)]
        ev[0].record()
        out = x
        for l in layers:
            out = l(out)
        ev[1].record()
        loss = out.float().square().mean()
        ev[2].record()
        loss.backward()
        ev[3].record()
        opt.step()
        opt.zero_grad(set_to_none=True)
        x.grad = None
        ev[4].record()
        torch.cuda.synchronize()
        if i >= warmup:
            fwd.append(ev[0].elapsed_time(ev[2]))
            bwd.append(ev[2].elapsed_time(ev[3]))
            optt.append(ev[3].elapsed_time(ev[4]))

    def stat(v):
        return {"mean_ms": statistics.fmean(v),
                "std_ms": statistics.pstdev(v) if len(v) > 1 else 0.0}
    return {"forward": stat(fwd), "backward": stat(bwd), "optimizer": stat(optt)}


In [ ]:
results = {}
for name, cfg in CONFIGS.items():
    r = bench(cfg["dims"], cfg["tokens"], ITERS, WARMUP)
    results[name] = {"dims": cfg["dims"], "tokens": cfg["tokens"], **r}
    print(f"  {name:16s}  fwd {r['forward']['mean_ms']:8.3f}  "
          f"bwd {r['backward']['mean_ms']:8.3f}  "
          f"opt {r['optimizer']['mean_ms']:6.3f} ms")


In [ ]:
import json, platform

out = {
    "gpu": gpu, "torch": torch.__version__, "cuda": torch.version.cuda,
    "dtype": "float16", "iters": ITERS, "warmup": WARMUP,
    "configs": results, "host": platform.platform(),
}
with open(OUT, "w") as f:
    json.dump(out, f, indent=2)

print(f"\n===== {OUT} (copy this back if the download fails) =====\n")
print(json.dumps(out, indent=2))

try:
    from google.colab import files
    files.download(OUT)
except Exception as e:
    print(f"\n(auto-download unavailable: {e} -- grab {OUT} from the Files sidebar)")
